# Módulo 4 · Práctica: comprobar gradientes

Entrenar un LLM es bajar una montaña en un espacio de miles de millones de dimensiones, guiándose solo por el gradiente. Si el gradiente está mal, el modelo baja hacia donde no debe y nadie lo nota a simple vista. En este notebook aprendes la herramienta que usan los ingenieros para cazar esos errores: la **comprobación numérica con diferencias finitas centradas**.

**Cómo trabajar:** ejecuta las celdas en orden. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que va detrás.

| Parte | Qué haces |
| --- | --- |
| 1 | Gradiente numérico con diferencias centradas |
| 2 | Error relativo y la elección de h |
| 3 | Softmax + entropía cruzada: el gradiente p − y |
| 4 | Capa lineal: ∂L/∂W, ∂L/∂X, ∂L/∂b |
| 5 | Juntarlo todo: regresión softmax y descenso de gradiente |
| 6 | Hessiano numérico y convexidad |
| 7 | Retos |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## Parte 1 · Gradiente numérico

Para cada coordenada $i$ movemos el punto $\pm h$ y medimos la pendiente:

$$\frac{\partial f}{\partial x_i} \approx \frac{f(\mathbf{x} + h\,\mathbf{e}_i) - f(\mathbf{x} - h\,\mathbf{e}_i)}{2h}$$

Debe servir para **arrays de cualquier forma**, porque los parámetros de una red son matrices. Truco: recorrer los índices con `np.ndindex(x.shape)`, modificar `x[idx]`, evaluar y **restaurar** el valor.

**Ejercicio 1.** Completa `grad_num(f, x, h)`. Debe devolver un array con la misma forma que `x` y dejar `x` sin modificar.

In [ ]:
def grad_num(f, x, h=1e-5):
    x = x.astype(float)          # copia en coma flotante
    g = np.zeros_like(x)
    for idx in np.ndindex(x.shape):
        # TU CÓDIGO: guarda x[idx], evalúa f en x[idx] + h y x[idx] - h,
        # restaura x[idx] y guarda la diferencia centrada en g[idx]
        pass
    return g

In [ ]:
# Comprobación: el ejemplo del documento, f(x, y) = x²y + 3y en (2, 1) -> (4, 7)
f = lambda v: v[0]**2 * v[1] + 3 * v[1]
p0 = np.array([2.0, 1.0])
g = grad_num(f, p0)
assert g.shape == (2,), "grad_num debe devolver la forma de x"
assert np.allclose(g, [4, 7], atol=1e-6), f"Esperaba (4, 7) y sale {g}"
assert np.allclose(p0, [2, 1]), "grad_num no debe modificar el punto de entrada"
G = grad_num(lambda M: (M**2).sum(), np.arange(6.0).reshape(2, 3))
assert G.shape == (2, 3) and np.allclose(G, 2 * np.arange(6.0).reshape(2, 3)), "Debe funcionar con matrices"
print("Correcto. ∇f(2, 1) ≈", g)

**Ejemplo de la cadena (sección 4.3).** g(x, y) = (xy, x² + y) y f(u, v) = u²v. El documento obtiene $J^\top \nabla_u f = (36, 44)$. Compruébalo:

In [ ]:
def compuesta(v):
    x, y = v
    u, w = x * y, x**2 + y
    return u**2 * w

J = np.array([[1., 2.], [4., 1.]])      # jacobiano de g en (2, 1)
grad_ext = np.array([20., 4.])          # (∂f/∂u, ∂f/∂v) en (u, v) = (2, 5)
print("Jᵀ·∇f      :", J.T @ grad_ext)
print("Numérico   :", grad_num(compuesta, np.array([2., 1.])))

## Parte 2 · Error relativo y elección de h

Para decidir si un gradiente analítico $a$ coincide con uno numérico $n$ usamos el error relativo, que no depende de la escala:

$$e = \frac{\lVert a - n \rVert}{\max(\lVert a \rVert + \lVert n \rVert,\ 10^{-12})}$$

Regla práctica en fp64: $e < 10^{-7}$ es excelente; $e > 10^{-3}$ indica casi seguro un error.

**Ejercicio 2.** Implementa `error_rel(a, n)` con la norma euclídea (`np.linalg.norm`).

In [ ]:
def error_rel(a, n):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
assert error_rel(np.array([3., 4.]), np.array([3., 4.])) == 0, "Vectores iguales -> 0"
assert abs(error_rel(np.array([1., 0.]), np.array([0., 1.])) - np.sqrt(2) / 2) < 1e-12, "Revisa la fórmula"
assert error_rel(np.zeros(3), np.zeros(3)) == 0, "Cuidado con dividir entre cero"
print("Correcto")

Ahora mide cómo depende el error de $h$ para $f(x) = e^x$ en $x = 0$ (derivada exacta 1), con la diferencia hacia delante y la centrada. Verás las dos pendientes (orden $h$ y $h^2$) y, para $h$ muy pequeño, cómo el redondeo de la coma flotante lo estropea todo.

In [ ]:
hs = np.logspace(-12, -1, 45)
err_fw = [abs((np.exp(h) - 1) / h - 1) for h in hs]
err_ce = [abs((np.exp(h) - np.exp(-h)) / (2 * h) - 1) for h in hs]

for h in [1e-1, 1e-2, 1e-3]:
    print(f"h = {h:.0e}  delante {abs((np.exp(h) - 1) / h - 1):.2e}  centrada {abs((np.exp(h) - np.exp(-h)) / (2 * h) - 1):.2e}")

plt.loglog(hs, err_fw, label="hacia delante, O(h)")
plt.loglog(hs, err_ce, label="centrada, O(h²)")
plt.xlabel("h"); plt.ylabel("error absoluto"); plt.legend(); plt.grid(alpha=.3)
plt.title("Truncamiento frente a redondeo"); plt.show()
print("h con menor error centrado:", f"{hs[np.argmin(err_ce)]:.1e}")

**Pregunta.** ¿Por qué el mínimo de la curva centrada está entre $10^{-6}$ y $10^{-5}$, y el de la curva hacia delante cerca de $10^{-8}$? (Pista: $\varepsilon^{1/3}$ y $\varepsilon^{1/2}$ con $\varepsilon \approx 10^{-16}$.)

## Parte 3 · Softmax + entropía cruzada

Con logits $\mathbf{z}$, $\mathbf{p} = \mathrm{softmax}(\mathbf{z})$ y token correcto $c$:

$$L = -\log p_c, \qquad \nabla_{\mathbf{z}} L = \mathbf{p} - \mathbf{y}$$

Para un **lote** de $B$ ejemplos, $Z$ tiene forma $(B, K)$, la pérdida es la media y el gradiente es $(P - Y)/B$. Calcularemos la pérdida de forma estable con log-sum-exp: $\log\sum_k e^{z_k} = m + \log\sum_k e^{z_k - m}$ con $m = \max_k z_k$.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=-1, keepdims=True)

def ce_loss(Z, c):
    """Entropía cruzada media. Z: (B, K) logits; c: (B,) índices correctos."""
    m = Z.max(axis=1, keepdims=True)
    lse = (m + np.log(np.exp(Z - m).sum(axis=1, keepdims=True))).ravel()
    return np.mean(lse - Z[np.arange(len(c)), c])

print(softmax(np.array([2., 1., 0.])))      # documento: 0,665 0,245 0,090

**Ejercicio 3.** Implementa `ce_grad(Z, c)` = $(P - Y)/B$ sin bucles. Pista: construye `Y` one-hot con `np.eye(K)[c]`.

In [ ]:
def ce_grad(Z, c):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación 1: ejemplo del documento, z = (2, 1, 0), correcto el primero
g1 = ce_grad(np.array([[2., 1., 0.]]), np.array([0]))
assert g1 is not None and g1.shape == (1, 3), "La forma debe ser (B, K)"
assert np.allclose(g1, [[-0.334759, 0.244728, 0.090031]], atol=1e-6), f"Sale {g1}"
assert abs(ce_loss(np.array([[2., 1., 0.]]), np.array([0])) - 0.407606) < 1e-6

# Comprobación 2: lote aleatorio contra el gradiente numérico
rng = np.random.default_rng(0)
Z = rng.normal(size=(5, 7)); c = rng.integers(0, 7, size=5)
e = error_rel(ce_grad(Z, c), grad_num(lambda Zm: ce_loss(Zm, c), Z))
assert e < 1e-7, f"Error relativo demasiado grande: {e:.2e}"
assert np.allclose(ce_grad(Z, c).sum(axis=1), 0), "Cada fila de p - y debe sumar 0"
print(f"Correcto. Error relativo analítico vs numérico: {e:.2e}")

**Experimento: un gradiente con un error sutil.** Un fallo típico es olvidar dividir entre $B$. La comprobación lo detecta al instante:

In [ ]:
mal = lambda Z, c: softmax(Z) - np.eye(Z.shape[1])[c]     # falta /B
print(f"Error relativo con el fallo: {error_rel(mal(Z, c), grad_num(lambda Zm: ce_loss(Zm, c), Z)):.3f}")

## Parte 4 · Capa lineal

Trabajamos por lotes con filas: $X$ es $(B, n)$, $W$ es $(n, m)$, $\mathbf{b}$ es $(m,)$ y $Z = XW + \mathbf{b}$. Si llega $\Delta = \partial L/\partial Z$ de forma $(B, m)$:

$$\frac{\partial L}{\partial W} = X^\top \Delta, \qquad \frac{\partial L}{\partial X} = \Delta W^\top, \qquad \frac{\partial L}{\partial \mathbf{b}} = \sum_{\text{filas}} \Delta$$

Es la versión transpuesta de $\partial L/\partial W = \boldsymbol{\delta}\mathbf{x}^\top$ del documento. Regla de oro: cada gradiente tiene la forma de su variable.

**Ejercicio 4.** Completa `lineal_backward(X, W, dZ)`, que devuelve `(dX, dW, db)`.

In [ ]:
def lineal_forward(X, W, b):
    return X @ W + b

In [ ]:
def lineal_backward(X, W, dZ):
    # TU CÓDIGO
    dX = dW = db = None
    return dX, dW, db

In [ ]:
# Comprobación 1: ejemplo del documento (un solo ejemplo, con y = Wx -> aquí Z = x Wᵀ)
Wd = np.array([[1., 2.], [0., 1.]]); x = np.array([[1., -1.]]); t = np.array([[0., 1.]])
Zd = lineal_forward(x, Wd.T, np.zeros(2))       # = (Wd x)ᵀ = [[-1, -1]]
dXd, dWd, _ = lineal_backward(x, Wd.T, Zd - t)  # pérdida ½‖z - t‖² -> dZ = z - t
assert dWd is not None, "Completa lineal_backward"
assert np.allclose(dWd.T, [[-1, 1], [-2, 2]]), "∂L/∂W del documento: [[-1, 1], [-2, 2]]"
assert np.allclose(dXd, [[-1, -4]]), "∂L/∂x del documento: (-1, -4)"

# Comprobación 2: lote aleatorio con pérdida L = Σ (Z ⊙ R), cuyo ∂L/∂Z = R
rng = np.random.default_rng(1)
X = rng.normal(size=(4, 3)); W = rng.normal(size=(3, 5)); b = rng.normal(size=5)
R = rng.normal(size=(4, 5))
L = lambda X_, W_, b_: np.sum(lineal_forward(X_, W_, b_) * R)
dX, dW, db = lineal_backward(X, W, R)
assert dX.shape == X.shape and dW.shape == W.shape and db.shape == b.shape, "Formas incorrectas"
for nombre, an, nu in [("dX", dX, grad_num(lambda A: L(A, W, b), X)),
                       ("dW", dW, grad_num(lambda A: L(X, A, b), W)),
                       ("db", db, grad_num(lambda A: L(X, W, A), b))]:
    e = error_rel(an, nu)
    assert e < 1e-7, f"{nombre}: error relativo {e:.2e}"
    print(f"{nombre}: error relativo {e:.1e}")
print("Correcto")

## Parte 5 · Regresión softmax: bajar la montaña

Unimos las partes 3 y 4: una capa lineal produce logits y la entropía cruzada los evalúa. Es exactamente la última capa de un LLM: $\mathbf{h}$ (flujo residual) → logits sobre el vocabulario → pérdida. Usamos datos sintéticos: 3 clases de puntos en el plano.

In [ ]:
rng = np.random.default_rng(42)
centros = np.array([[0., 2.], [-2., -1.], [2., -1.]])
c_data = rng.integers(0, 3, size=150)
X_data = centros[c_data] + rng.normal(scale=0.9, size=(150, 2))
plt.scatter(X_data[:, 0], X_data[:, 1], c=c_data, cmap="viridis", s=15)
plt.title("Datos sintéticos: 3 clases"); plt.axis("equal"); plt.show()

**Ejercicio 5.** Completa `perdida_y_grads(W, b, X, c)`: hacia delante (lineal + CE) y hacia atrás (`ce_grad` y `lineal_backward`). Devuelve `(L, dW, db)`.

In [ ]:
def perdida_y_grads(W, b, X, c):
    # TU CÓDIGO
    L, dW, db = None, None, None
    return L, dW, db

In [ ]:
# Comprobación: gradiente analítico frente a numérico antes de entrenar
W0 = rng.normal(scale=0.1, size=(2, 3)); b0 = np.zeros(3)
L0, dW0, db0 = perdida_y_grads(W0, b0, X_data, c_data)
assert L0 is not None, "Completa perdida_y_grads"
eW = error_rel(dW0, grad_num(lambda A: perdida_y_grads(A, b0, X_data, c_data)[0], W0))
eb = error_rel(db0, grad_num(lambda A: perdida_y_grads(W0, A, X_data, c_data)[0], b0))
assert eW < 1e-7 and eb < 1e-7, f"Errores relativos: W {eW:.1e}, b {eb:.1e}"
print(f"Correcto. Pérdida inicial {L0:.3f} (ln 3 = {np.log(3):.3f}); errores {eW:.1e}, {eb:.1e}")

Ahora sí, descenso de gradiente: $\theta \leftarrow \theta - \eta\,\nabla L$.

In [ ]:
W, b, eta = W0.copy(), b0.copy(), 0.5
historia = []
for paso in range(200):
    L, dW, db = perdida_y_grads(W, b, X_data, c_data)
    W -= eta * dW
    b -= eta * db
    historia.append(L)

acierto = np.mean(np.argmax(lineal_forward(X_data, W, b), axis=1) == c_data)
print(f"Pérdida: {historia[0]:.3f} -> {historia[-1]:.3f}   acierto: {acierto:.0%}")
plt.plot(historia); plt.xlabel("paso"); plt.ylabel("entropía cruzada (nats)")
plt.title("Bajando la montaña"); plt.grid(alpha=.3); plt.show()
assert historia[-1] < historia[0] - 0.3, "La pérdida debería bajar claramente"
assert acierto > 0.8, "La regresión softmax debería acertar más del 80 %" 

**Pregunta.** Este problema es convexo (ejercicio 14 del documento). ¿Qué cambiaría si añadiéramos una capa oculta con ReLU? Prueba η = 5 y η = 0,05: ¿qué observas?

## Parte 6 · Hessiano numérico y convexidad

El hessiano es el jacobiano del gradiente. Podemos estimarlo aplicando `grad_num` a cada componente del gradiente.

**Ejercicio 6.** Completa `hess_num(f, x)` para `x` vectorial: la columna `j` es la diferencia centrada de `grad_num(f, ·)` en la dirección $\mathbf{e}_j$.

In [ ]:
def hess_num(f, x, h=1e-4):
    n = x.size
    H = np.zeros((n, n))
    # TU CÓDIGO
    return H

In [ ]:
# Comprobación: f(x, y) = x²y + 3y en (2, 1) -> H = [[2, 4], [4, 0]]
H = hess_num(f, np.array([2., 1.]))
assert np.allclose(H, [[2, 4], [4, 0]], atol=1e-4), f"Sale {H}"
print("H ≈\n", H.round(4), "\nautovalores:", np.linalg.eigvalsh((H + H.T) / 2))   # 1 ± √17: silla

# Hessiano de la entropía cruzada en los logits z = (2, 1, 0): diag(p) - ppᵀ
z = np.array([2., 1., 0.])
Hce = hess_num(lambda v: ce_loss(v[None, :], np.array([0])), z)
p = softmax(z)
assert np.allclose(Hce, np.diag(p) - np.outer(p, p), atol=1e-4), "Debe coincidir con diag(p) - ppᵀ"
print("autovalores del hessiano de la CE:", np.linalg.eigvalsh(np.diag(p) - np.outer(p, p)).round(3))
print("Correcto: todos ≥ 0, la entropía cruzada es convexa en los logits")

Comprueba también la aproximación de Taylor del documento: en $(2, 1)$ con $\mathbf{h} = (0{,}1;\ -0{,}1)$, orden 1 da 6,7, orden 2 da 6,67 y el valor exacto es 6,669.

In [ ]:
x0, hv = np.array([2., 1.]), np.array([0.1, -0.1])
g0 = grad_num(f, x0)
print("orden 1:", round(f(x0) + g0 @ hv, 4))
print("orden 2:", round(f(x0) + g0 @ hv + 0.5 * hv @ H @ hv, 4))
print("exacto :", round(f(x0 + hv), 4))

## Parte 7 · Retos

1. **Capa oculta.** Añade una capa oculta con tanh entre la entrada y los logits: $Z = \tanh(XW_1 + b_1)W_2 + b_2$. Deriva el gradiente (la derivada de tanh es $1 - \tanh^2$) y compruébalo con `grad_num`. Entrena y compara la pérdida con la parte 5.
2. **Comprobación por muestreo.** Para una matriz grande (por ejemplo 200 × 300), comprueba solo 20 coordenadas sorteadas. ¿Por qué es lo que se hace en modelos reales?
3. **Derivada direccional.** Con un vector aleatorio $\mathbf{u}$, compara $\nabla L \cdot \mathbf{u}$ con $[L(\theta + h\mathbf{u}) - L(\theta - h\mathbf{u})]/2h$. Solo cuesta 2 evaluaciones sea cual sea el número de parámetros.
4. **Precisión.** Repite la comprobación 2 de la parte 3 convirtiendo `Z` a `np.float32`. ¿Qué error relativo obtienes? ¿Por qué no se comprueban gradientes en fp16?

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
# def grad_num(f, x, h=1e-5):
#     x = x.astype(float)
#     g = np.zeros_like(x)
#     for idx in np.ndindex(x.shape):
#         viejo = x[idx]
#         x[idx] = viejo + h; fp = f(x)
#         x[idx] = viejo - h; fm = f(x)
#         x[idx] = viejo
#         g[idx] = (fp - fm) / (2 * h)
#     return g

# Ejercicio 2
# def error_rel(a, n):
#     a, n = np.asarray(a, float), np.asarray(n, float)
#     return np.linalg.norm(a - n) / max(np.linalg.norm(a) + np.linalg.norm(n), 1e-12)

# Ejercicio 3
# def ce_grad(Z, c):
#     B, K = Z.shape
#     return (softmax(Z) - np.eye(K)[c]) / B

# Ejercicio 4
# def lineal_backward(X, W, dZ):
#     dX = dZ @ W.T
#     dW = X.T @ dZ
#     db = dZ.sum(axis=0)
#     return dX, dW, db

# Ejercicio 5
# def perdida_y_grads(W, b, X, c):
#     Z = lineal_forward(X, W, b)
#     L = ce_loss(Z, c)
#     _, dW, db = lineal_backward(X, W, ce_grad(Z, c))
#     return L, dW, db

# Ejercicio 6
# def hess_num(f, x, h=1e-4):
#     n = x.size
#     H = np.zeros((n, n))
#     for j in range(n):
#         e = np.zeros(n); e[j] = h
#         H[:, j] = (grad_num(f, x + e) - grad_num(f, x - e)) / (2 * h)
#     return H
